In [1]:
import os
import math
import time
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import IterableDataset, DataLoader

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

print(
    "Torch CPU threads:",
    torch.get_num_threads()
)

print(
    "Torch interop threads:",
    torch.get_num_interop_threads()
)

Torch CPU threads: 1
Torch interop threads: 1


In [2]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROCESSED_DIR = Path(
    r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_processed"
)

VECTORS_DIR = Path(
    r"D:\Lokesh files\Projects\LearningDNA\processed\kt1_vectors"
)

MODEL_DIR = Path(
    r"D:\Lokesh files\Projects\LearningDNA\models"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(props.total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.8.0+cu129
Device: cuda
GPU: NVIDIA GeForce MX450
VRAM: 2.0 GB


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [4]:
# ============================================================
# FEATURE DIMENSIONS
# ============================================================

QUESTION_DIM = 64
SKILL_DIM = 32
CORRECTNESS_DIM = 8
ELAPSED_DIM = 16
ATTEMPT_DIM = 8
TIME_DELTA_DIM = 8

MODEL_DIM = 192

# ============================================================
# TRANSFORMER
# ============================================================

NUM_HEADS = 6
NUM_LAYERS = 3
FF_DIM = 384
DROPOUT = 0.1

# ============================================================
# SEQUENCE
# ============================================================

SEQ_LEN = 128
STRIDE = 64

# ============================================================
# TRAINING
# ============================================================

BATCH_SIZE = 64

NUM_WORKERS = 0

# Effective batch size:
# BATCH_SIZE × GRAD_ACCUM_STEPS

GRAD_ACCUM_STEPS = 1

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 3

# ============================================================
# DATA SPLIT
# ============================================================

VAL_MOD = 10

# user_id % 10 == 0 → validation
# everything else → training

# ============================================================
# CHECKPOINTING
# ============================================================

SAVE_EVERY_STEPS = 500

SAVE_EVERY_MINUTES = 30

# ============================================================
# PILOT
# ============================================================

PILOT_PARTS = 50

# ============================================================
# PERFORMANCE
# ============================================================

USE_AMP = DEVICE.type == "cuda"

print("Configuration loaded.")
print("Model dimension:", MODEL_DIM)
print("Transformer layers:", NUM_LAYERS)
print("Attention heads:", NUM_HEADS)
print("Feed-forward dimension:", FF_DIM)
print("Sequence length:", SEQ_LEN)
print("Stride:", STRIDE)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Workers:", NUM_WORKERS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("AMP:", USE_AMP)

Configuration loaded.
Model dimension: 192
Transformer layers: 3
Attention heads: 6
Feed-forward dimension: 384
Sequence length: 128
Stride: 64
Batch size: 64
Gradient accumulation: 1
Effective batch size: 64
Workers: 0
Learning rate: 0.0003
Weight decay: 0.0001
AMP: True


In [5]:
QUESTION_VOCAB_FILE = (
    VECTORS_DIR / "question_vocab.pkl"
)

SKILL_VOCAB_FILE = (
    VECTORS_DIR / "skill_vocab.pkl"
)

with open(QUESTION_VOCAB_FILE, "rb") as f:
    question_to_idx = pickle.load(f)

with open(SKILL_VOCAB_FILE, "rb") as f:
    skill_to_idx = pickle.load(f)

NUM_QUESTIONS = len(question_to_idx)
NUM_SKILLS = len(skill_to_idx)

print("Number of questions:", NUM_QUESTIONS)
print("Number of skills:", NUM_SKILLS)

Number of questions: 12284
Number of skills: 190


In [6]:
QUESTIONS_FILE = Path(
    r"D:\Lokesh files\Projects\LearningDNA\contents\questions.csv"
)

questions_df = pd.read_csv(
    QUESTIONS_FILE
)

print(
    "questions.csv rows:",
    len(questions_df)
)

print(
    "Columns:",
    questions_df.columns.tolist()
)

question_skill_map = {}

for _, row in questions_df.iterrows():

    qid = str(row["question_id"]).strip()

    tags = row["tags"]

    if pd.isna(tags):
        question_skill_map[qid] = []
        continue

    skills = []

    for tag in str(tags).split(";"):

        tag = tag.strip()

        if tag in skill_to_idx:
            skills.append(
                skill_to_idx[tag]
            )

    question_skill_map[qid] = skills

print(
    "Question-skill mappings:",
    len(question_skill_map)
)

questions.csv rows: 13169
Columns: ['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']
Question-skill mappings: 13169


In [7]:
# ============================================================
# PRECOMPUTE QUESTION → SKILL TENSOR
# ============================================================

MAX_SKILLS = 4

question_skill_tensor = torch.full(
    (
        NUM_QUESTIONS,
        MAX_SKILLS
    ),
    -1,
    dtype=torch.long
)

for question_id, question_idx in question_to_idx.items():

    skills = question_skill_map.get(
        question_id,
        []
    )

    for j, skill_idx in enumerate(
        skills[:MAX_SKILLS]
    ):
        question_skill_tensor[
            question_idx,
            j
        ] = skill_idx

print(
    "Question-skill tensor shape:",
    tuple(question_skill_tensor.shape)
)

print(
    "Example q1 index:",
    question_to_idx.get("q1")
)

q1_idx = question_to_idx.get("q1")

if q1_idx is not None:
    print(
        "q1 skills:",
        question_skill_tensor[q1_idx].tolist()
    )

Question-skill tensor shape: (12284, 4)
Example q1 index: 0
q1 skills: [2, 100, 88, 91]


In [8]:
class LearningDNA(nn.Module):

    def __init__(
        self,
        num_questions,
        num_skills,
        question_dim=64,
        skill_dim=32,
        correctness_dim=8,
        elapsed_dim=16,
        attempt_dim=8,
        time_delta_dim=8,
        model_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.1,
        max_seq_len=64
    ):
        super().__init__()

        self.model_dim = model_dim

        # ----------------------------------------------------
        # Question embedding
        # ----------------------------------------------------

        self.question_embedding = nn.Embedding(
            num_questions,
            question_dim
        )

        # 0 = incorrect
        # 1 = correct
        # 2 = missing

        self.correctness_embedding = nn.Embedding(
            3,
            correctness_dim
        )

        # Attempt 1...256
        self.attempt_embedding = nn.Embedding(
            256,
            attempt_dim
        )

        # ----------------------------------------------------
        # Skill embedding
        # ----------------------------------------------------

        self.skill_embedding = nn.Embedding(
            num_skills,
            skill_dim
        )

        # ----------------------------------------------------
        # Continuous features
        # ----------------------------------------------------

        self.elapsed_projection = nn.Sequential(
            nn.Linear(1, elapsed_dim),
            nn.ReLU()
        )

        self.time_delta_projection = nn.Sequential(
            nn.Linear(1, time_delta_dim),
            nn.ReLU()
        )

        # ----------------------------------------------------
        # Feature fusion
        # ----------------------------------------------------

        input_dim = (
            question_dim
            + skill_dim
            + correctness_dim
            + elapsed_dim
            + attempt_dim
            + time_delta_dim
        )

        self.feature_projection = nn.Linear(
            input_dim,
            model_dim
        )

        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            max_seq_len,
            model_dim
        )

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # ----------------------------------------------------
        # Prediction head
        # ----------------------------------------------------

        self.prediction_head = nn.Sequential(
            nn.Linear(
                model_dim + question_dim,
                128
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        question_idx,
        skill_idx,
        correctness_idx,
        elapsed,
        attempt_idx,
        time_delta,
        target_question_idx
    ):

        # ----------------------------------------------------
        # Question
        # ----------------------------------------------------

        q = self.question_embedding(
            question_idx
        )

        # ----------------------------------------------------
        # Skills
        # ----------------------------------------------------

        skill_emb = self.skill_embedding(
            skill_idx.clamp(min=0)
        )

        valid_skill = (
            skill_idx >= 0
        ).unsqueeze(-1)

        skill_emb = (
            skill_emb * valid_skill
        )

        skill_count = (
            valid_skill.sum(dim=2)
            .clamp(min=1)
        )

        skill_emb = (
            skill_emb.sum(dim=2)
            / skill_count
        )

        # ----------------------------------------------------
        # Correctness
        # ----------------------------------------------------

        c = self.correctness_embedding(
            correctness_idx
        )

        # ----------------------------------------------------
        # Elapsed time
        # ----------------------------------------------------

        e = self.elapsed_projection(
            elapsed
        )

        # ----------------------------------------------------
        # Attempt
        # ----------------------------------------------------

        a = self.attempt_embedding(
            attempt_idx.clamp(
                min=0,
                max=255
            )
        )

        # ----------------------------------------------------
        # Time delta
        # ----------------------------------------------------

        td = self.time_delta_projection(
            time_delta
        )

        # ----------------------------------------------------
        # Combine features
        # ----------------------------------------------------

        x = torch.cat(
            [
                q,
                skill_emb,
                c,
                e,
                a,
                td
            ],
            dim=-1
        )

        x = self.feature_projection(
            x
        )

        # ----------------------------------------------------
        # Position
        # ----------------------------------------------------

        positions = torch.arange(
            x.size(1),
            device=x.device
        )

        x = (
            x
            + self.position_embedding(
                positions
            )
        )

        x = self.dropout(x)

        # ----------------------------------------------------
        # Causal attention mask
        # ----------------------------------------------------

        seq_len = x.size(1)

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        h = self.transformer(
            x,
            mask=causal_mask
        )

        # ----------------------------------------------------
        # Next question
        # ----------------------------------------------------

        next_q = self.question_embedding(
            target_question_idx
        )

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        combined = torch.cat(
            [
                h,
                next_q
            ],
            dim=-1
        )

        logits = self.prediction_head(
            combined
        ).squeeze(-1)

        return logits, h

In [9]:
model = LearningDNA(
    num_questions=NUM_QUESTIONS,
    num_skills=NUM_SKILLS,
    question_dim=QUESTION_DIM,
    skill_dim=SKILL_DIM,
    correctness_dim=CORRECTNESS_DIM,
    elapsed_dim=ELAPSED_DIM,
    attempt_dim=ATTEMPT_DIM,
    time_delta_dim=TIME_DELTA_DIM,
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
    max_seq_len=SEQ_LEN
).to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Total parameters: {total_params:,}"
)

print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

D:\Lokesh files\Projects\LearningDNA\venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Total parameters: 1,769,353
Trainable parameters: 1,769,353


In [10]:
PART_FILES = sorted(
    PROCESSED_DIR.glob("part_*.parquet")
)

print(
    "Parquet parts:",
    len(PART_FILES)
)

TRAIN_COLUMNS = [
    "user_id",
    "question_id",
    "correctness",
    "answer_missing",
    "log_elapsed_time",
    "attempt_index",
    "log_time_delta"
]

Parquet parts: 15687


In [11]:
# ============================================================
# KT SEQUENCE DATASET - PART-LEVEL RESUMABLE
# ============================================================

class KTSequenceDataset(IterableDataset):

    def __init__(
        self,
        part_files,
        train=True,
        seq_len=64,
        stride=32,
        max_parts=None,
        start_part=0
    ):
        self.part_files = part_files
        self.train = train
        self.seq_len = seq_len
        self.stride = stride
        self.max_parts = max_parts
        self.start_part = start_part

    def student_is_train(self, user_id):
        return int(user_id) % VAL_MOD != 0

    def make_samples(self, group, part_index):

        question_ids = (
            group["question_id"]
            .astype(str)
            .to_numpy()
        )

        correctness = group[
            "correctness"
        ].to_numpy(dtype=np.int64)

        missing = group[
            "answer_missing"
        ].to_numpy(dtype=np.int64)

        elapsed = group[
            "log_elapsed_time"
        ].to_numpy(dtype=np.float32)

        attempt = group[
            "attempt_index"
        ].to_numpy(dtype=np.int64)

        time_delta = group[
            "log_time_delta"
        ].to_numpy(dtype=np.float32)

        n = len(group)

        if n < 2:
            return

        start = 0

        while start < n - 1:

            end = min(
                start + self.seq_len,
                n - 1
            )

            if end <= start:
                break

            current_q = question_ids[start:end]

            target_q = question_ids[
                start + 1:end + 1
            ]

            current_correctness = (
                correctness[start:end].copy()
            )

            current_correctness[
                missing[start:end] == 1
            ] = 2

            target_correctness = (
                correctness[start + 1:end + 1].copy()
            )

            target_missing = (
                missing[start + 1:end + 1]
            )

            target_mask = (
                target_missing == 0
            )

            if target_mask.sum() == 0:
                start += self.stride
                continue

            current_q_idx = np.array(
                [
                    question_to_idx.get(q, 0)
                    for q in current_q
                ],
                dtype=np.int64
            )

            target_q_idx = np.array(
                [
                    question_to_idx.get(q, 0)
                    for q in target_q
                ],
                dtype=np.int64
            )

            skill_array = (
                question_skill_tensor[
                    torch.from_numpy(
                        current_q_idx
                    )
                ].numpy()
            )

            attempt_idx = np.clip(
                attempt[start:end] - 1,
                0,
                255
            )

            yield {
                "question_idx": current_q_idx,
                "skill_idx": skill_array,
                "correctness_idx": current_correctness,
                "elapsed": elapsed[start:end],
                "attempt_idx": attempt_idx,
                "time_delta": time_delta[start:end],
                "target_question_idx": target_q_idx,
                "target": target_correctness.astype(
                    np.float32
                ),
                "target_mask": target_mask,
                "part_index": part_index
            }

            start += self.stride

            if end == n - 1:
                break

    def __iter__(self):

        from torch.utils.data import get_worker_info

        files = self.part_files

        if self.max_parts is not None:
            files = files[:self.max_parts]

        worker_info = get_worker_info()

        if worker_info is not None:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            files = files[
                worker_id::num_workers
            ]

        files = files[self.start_part:]

        for local_index, part_file in enumerate(files):

            actual_part_index = (
                self.start_part
                + local_index
            )

            df = pd.read_parquet(
                part_file,
                columns=TRAIN_COLUMNS
            )

            df = df.sort_values(
                ["user_id"],
                kind="stable"
            )

            for user_id, group in df.groupby(
                "user_id",
                sort=False
            ):

                if (
                    self.student_is_train(user_id)
                    != self.train
                ):
                    continue

                yield from self.make_samples(
                    group,
                    actual_part_index
                )

            del df

In [12]:
# ============================================================
# COLLATE FUNCTION
# ============================================================

def collate_fn(samples):

    max_len = max(
        len(x["question_idx"])
        for x in samples
    )

    batch_size = len(samples)

    question_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    skill_idx = torch.full(
        (batch_size, max_len, 4),
        -1,
        dtype=torch.long
    )

    correctness_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    elapsed = torch.zeros(
        batch_size,
        max_len,
        1,
        dtype=torch.float32
    )

    attempt_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    time_delta = torch.zeros(
        batch_size,
        max_len,
        1,
        dtype=torch.float32
    )

    target_question_idx = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.long
    )

    target = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.float32
    )

    target_mask = torch.zeros(
        batch_size,
        max_len,
        dtype=torch.bool
    )

    part_indices = torch.zeros(
        batch_size,
        dtype=torch.long
    )

    for i, sample in enumerate(samples):

        length = len(
            sample["question_idx"]
        )

        question_idx[i, :length] = torch.from_numpy(
            sample["question_idx"]
        )

        skill_idx[i, :length] = torch.from_numpy(
            sample["skill_idx"]
        )

        correctness_idx[i, :length] = torch.from_numpy(
            sample["correctness_idx"]
        )

        elapsed[i, :length, 0] = torch.from_numpy(
            sample["elapsed"]
        )

        attempt_idx[i, :length] = torch.from_numpy(
            sample["attempt_idx"]
        )

        time_delta[i, :length, 0] = torch.from_numpy(
            sample["time_delta"]
        )

        target_question_idx[i, :length] = torch.from_numpy(
            sample["target_question_idx"]
        )

        target[i, :length] = torch.from_numpy(
            sample["target"]
        )

        target_mask[i, :length] = torch.from_numpy(
            sample["target_mask"]
        )

        part_indices[i] = sample["part_index"]

    return {
        "question_idx": question_idx,
        "skill_idx": skill_idx,
        "correctness_idx": correctness_idx,
        "elapsed": elapsed,
        "attempt_idx": attempt_idx,
        "time_delta": time_delta,
        "target_question_idx": target_question_idx,
        "target": target,
        "target_mask": target_mask,
        "part_index": part_indices
    }

In [13]:
# ============================================================
# FULL TRAINING CHECKPOINT
# ============================================================

FULL_CHECKPOINT_DIR = (
    MODEL_DIR / "full_training_checkpoints"
)

FULL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LATEST_FULL_CHECKPOINT = (
    FULL_CHECKPOINT_DIR /
    "latest_full_training.pt"
)

BEST_FULL_CHECKPOINT = (
    FULL_CHECKPOINT_DIR /
    "best_model.pt"
)


def save_training_checkpoint(
    epoch,
    global_step,
    part_index,
    model,
    optimizer,
    scheduler,
    scaler,
    best_val_loss,
    total_train_time,
    mid_epoch=True,
    running_loss=0.0,
    running_count=0
):

    checkpoint = {

        "epoch": epoch,

        "global_step": global_step,

        "part_index": part_index,

        "mid_epoch": mid_epoch,

        "running_loss": running_loss,

        "running_count": running_count,

        "model_state":
            model.state_dict(),

        "optimizer_state":
            optimizer.state_dict(),

        "scheduler_state":
            scheduler.state_dict(),

        "scaler_state":
            scaler.state_dict(),

        "best_val_loss":
            best_val_loss,

        "total_train_time":
            total_train_time,

        "config": {

            "SEQ_LEN": SEQ_LEN,

            "STRIDE": STRIDE,

            "BATCH_SIZE": BATCH_SIZE,

            "GRAD_ACCUM_STEPS":
                GRAD_ACCUM_STEPS,

            "MODEL_DIM":
                MODEL_DIM,

            "NUM_HEADS":
                NUM_HEADS,

            "NUM_LAYERS":
                NUM_LAYERS,

            "FF_DIM":
                FF_DIM
        }
    }

    torch.save(
        checkpoint,
        LATEST_FULL_CHECKPOINT
    )

    return checkpoint

In [14]:
# ============================================================
# CHECKPOINT LOADER
# ============================================================

def load_training_checkpoint():

    if not LATEST_FULL_CHECKPOINT.exists():

        print(
            "No full-training checkpoint found."
        )

        return None

    print("=" * 60)
    print("FULL TRAINING CHECKPOINT FOUND")
    print("=" * 60)

    checkpoint = torch.load(
        LATEST_FULL_CHECKPOINT,
        map_location=DEVICE
    )

    print(
        f"Saved epoch       : "
        f"{checkpoint['epoch']}"
    )

    print(
        f"Saved global step : "
        f"{checkpoint['global_step']:,}"
    )

    print(
        f"Saved part        : "
        f"{checkpoint['part_index']:,}"
    )

    print(
        f"Mid-epoch         : "
        f"{checkpoint.get('mid_epoch', True)}"
    )

    return checkpoint

In [15]:
def move_batch_to_device(batch):

    return {
        key: value.to(
            DEVICE,
            non_blocking=True
        )
        for key, value in batch.items()
    }

In [16]:
# ============================================================
# TRAIN ONE EPOCH - PERIODIC MID-EPOCH CHECKPOINTS
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    epoch,
    global_step=0,
    checkpoint_interval_minutes=30,
    best_val_loss=float("inf"),
    total_train_time=0.0,
    running_loss_start=0.0,
    running_count_start=0
):

    model.train()

    running_loss = running_loss_start
    running_count = running_count_start

    epoch_start = time.time()
    last_checkpoint_time = time.time()

    optimizer.zero_grad(
        set_to_none=True
    )

    for step, batch in enumerate(loader):

        batch = move_batch_to_device(
            batch
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            logits, hidden = model(
                batch["question_idx"],
                batch["skill_idx"],
                batch["correctness_idx"],
                batch["elapsed"],
                batch["attempt_idx"],
                batch["time_delta"],
                batch["target_question_idx"]
            )

            loss_matrix = (
                F.binary_cross_entropy_with_logits(
                    logits,
                    batch["target"],
                    reduction="none"
                )
            )

            mask = batch["target_mask"]

            if mask.sum() == 0:
                continue

            loss = (
                loss_matrix[mask].mean()
                / GRAD_ACCUM_STEPS
            )

        scaler.scale(
            loss
        ).backward()

        if (
            (step + 1)
            % GRAD_ACCUM_STEPS
            == 0
        ):

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_step += 1

        valid_count = mask.sum().item()

        running_loss += (
            loss.item()
            * GRAD_ACCUM_STEPS
            * valid_count
        )

        running_count += valid_count

        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if step % 100 == 0:

            elapsed = (
                time.time()
                - epoch_start
            )

            avg_loss = (
                running_loss
                / max(running_count, 1)
            )

            print(
                f"\r"
                f"Epoch {epoch} | "
                f"Batch {step:,} | "
                f"Global step {global_step:,} | "
                f"Loss {avg_loss:.4f} | "
                f"Time {elapsed / 3600:.2f} h",
                end=""
            )

        # ----------------------------------------------------
        # PERIODIC CHECKPOINT
        # ----------------------------------------------------

        checkpoint_elapsed = (
            time.time()
            - last_checkpoint_time
        )

        if (
            checkpoint_elapsed
            >= SAVE_EVERY_MINUTES * 60
        ):

            # Use the earliest part represented
            # in this batch. On resume, that part
            # will be replayed rather than skipped.
            resume_part = int(
                batch["part_index"].min().item()
            )

            elapsed_total = (
                total_train_time
                + time.time()
                - epoch_start
            )

            save_training_checkpoint(
                epoch=epoch,
                global_step=global_step,
                part_index=resume_part,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                best_val_loss=best_val_loss,
                total_train_time=elapsed_total,
                mid_epoch=True,
                running_loss=running_loss,
                running_count=running_count
            )

            print()
            print()
            print("=" * 60)
            print("MID-EPOCH CHECKPOINT SAVED")
            print("=" * 60)
            print(
                f"Epoch       : {epoch}"
            )
            print(
                f"Global step : {global_step:,}"
            )
            print(
                f"Resume part : {resume_part:,}"
            )
            print(
                f"Checkpoint  : "
                f"{LATEST_FULL_CHECKPOINT}"
            )
            print(
                "You may safely interrupt the notebook now."
            )
            print()

            last_checkpoint_time = time.time()

    print()

    epoch_loss = (
        running_loss
        / max(running_count, 1)
    )

    return (
        epoch_loss,
        global_step,
        running_loss,
        running_count
    )

In [17]:
@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in loader:

        batch = move_batch_to_device(
            batch
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            logits, _ = model(
                batch["question_idx"],
                batch["skill_idx"],
                batch["correctness_idx"],
                batch["elapsed"],
                batch["attempt_idx"],
                batch["time_delta"],
                batch["target_question_idx"]
            )

        mask = batch["target_mask"]

        valid_logits = logits[mask]

        valid_targets = (
            batch["target"][mask]
        )

        if valid_targets.numel() == 0:
            continue

        loss = (
            F.binary_cross_entropy_with_logits(
                valid_logits,
                valid_targets
            )
        )

        predictions = (
            torch.sigmoid(
                valid_logits
            ) >= 0.5
        )

        total_correct += (
            predictions
            == valid_targets.bool()
        ).sum().item()

        total_count += (
            valid_targets.numel()
        )

        total_loss += (
            loss.item()
            * valid_targets.numel()
        )

    return {
        "loss": (
            total_loss
            / max(total_count, 1)
        ),
        "accuracy": (
            total_correct
            / max(total_count, 1)
        ),
        "count": total_count
    }

In [18]:
train_dataset = KTSequenceDataset(
    PART_FILES,
    train=True,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    max_parts=PILOT_PARTS
)

val_dataset = KTSequenceDataset(
    PART_FILES,
    train=False,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    max_parts=PILOT_PARTS
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Workers:", NUM_WORKERS)
print("Batch size:", BATCH_SIZE)
print("Train loader ready.")
print("Validation loader ready.")

Workers: 0
Batch size: 64
Train loader ready.
Validation loader ready.


In [19]:
# ============================================================
# Optimizer + Scheduler + AMP
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print("Optimizer ready.")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("AMP enabled:", USE_AMP)

Optimizer ready.
Learning rate: 0.0003
Weight decay: 0.0001
AMP enabled: True


In [20]:
def print_gpu_stats():

    if DEVICE.type != "cuda":
        return

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**2
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**2
    )

    max_allocated = (
        torch.cuda.max_memory_allocated()
        / 1024**2
    )

    print(
        f"GPU memory | "
        f"allocated: {allocated:.1f} MB | "
        f"reserved: {reserved:.1f} MB | "
        f"max: {max_allocated:.1f} MB"
    )

In [21]:
if DEVICE.type == "cuda":

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

print_gpu_stats()

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "Current allocated:",
        round(
            torch.cuda.memory_allocated() / 1024**2,
            2
        ),
        "MB"
    )

    print(
        "Current reserved:",
        round(
            torch.cuda.memory_reserved() / 1024**2,
            2
        ),
        "MB"
    )

GPU memory | allocated: 6.8 MB | reserved: 24.0 MB | max: 6.8 MB
CUDA available: True
GPU: NVIDIA GeForce MX450
Current allocated: 6.76 MB
Current reserved: 24.0 MB


In [22]:
print("Testing DataLoader...")

test_start = time.time()

test_batch = next(
    iter(train_loader)
)

print(
    "First batch loaded successfully."
)

for key, value in test_batch.items():
    print(
        f"{key:22s}",
        tuple(value.shape),
        value.dtype
    )

print(
    f"Batch loading time: "
    f"{time.time() - test_start:.2f} sec"
)

Testing DataLoader...
First batch loaded successfully.
question_idx           (64, 128) torch.int64
skill_idx              (64, 128, 4) torch.int64
correctness_idx        (64, 128) torch.int64
elapsed                (64, 128, 1) torch.float32
attempt_idx            (64, 128) torch.int64
time_delta             (64, 128, 1) torch.float32
target_question_idx    (64, 128) torch.int64
target                 (64, 128) torch.float32
target_mask            (64, 128) torch.bool
part_index             (64,) torch.int64
Batch loading time: 2.39 sec


In [23]:
# ============================================================
# LEARNINGDNA FULL TRAINING - FINAL RESUMABLE RUN
# ============================================================

print("=" * 60)
print("LEARNINGDNA FULL TRAINING")
print("=" * 60)

print(f"Total dataset parts : {len(PART_FILES):,}")
print(f"Training epochs     : {MAX_EPOCHS}")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Sequence length     : {SEQ_LEN}")
print(f"Stride              : {STRIDE}")
print(f"Model dimension     : {MODEL_DIM}")
print(f"Transformer layers  : {NUM_LAYERS}")
print(f"Attention heads     : {NUM_HEADS}")
print(f"Workers             : {NUM_WORKERS}")
print(f"AMP enabled         : {USE_AMP}")
print(
    f"Checkpoint interval : "
    f"{SAVE_EVERY_MINUTES} minutes"
)
print()

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

model = LearningDNA(
    num_questions=len(question_to_idx),
    num_skills=len(skill_to_idx),
    question_dim=QUESTION_DIM,
    skill_dim=SKILL_DIM,
    correctness_dim=CORRECTNESS_DIM,
    elapsed_dim=ELAPSED_DIM,
    attempt_dim=ATTEMPT_DIM,
    time_delta_dim=TIME_DELTA_DIM,
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
    max_seq_len=SEQ_LEN
).to(DEVICE)

print("Model created.")

print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

print()

# ------------------------------------------------------------
# OPTIMIZER
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# SCHEDULER
# ------------------------------------------------------------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS
)

# ------------------------------------------------------------
# AMP
# ------------------------------------------------------------

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

# ------------------------------------------------------------
# CHECKPOINT STATE
# ------------------------------------------------------------

start_epoch = 1
start_part = 0
global_step = 0
best_val_loss = float("inf")
previous_training_time = 0.0

resume_running_loss = 0.0
resume_running_count = 0

checkpoint = load_training_checkpoint()

if checkpoint is not None:

    saved_config = checkpoint.get(
        "config",
        {}
    )

    current_config = {
        "SEQ_LEN": SEQ_LEN,
        "STRIDE": STRIDE,
        "BATCH_SIZE": BATCH_SIZE,
        "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
        "MODEL_DIM": MODEL_DIM,
        "NUM_HEADS": NUM_HEADS,
        "NUM_LAYERS": NUM_LAYERS,
        "FF_DIM": FF_DIM
    }

    if any(
        saved_config.get(k) != v
        for k, v in current_config.items()
    ):

        raise RuntimeError(
            "Checkpoint configuration does not "
            "match the current configuration."
        )

    model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state"]
    )

    if "scaler_state" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler_state"]
        )

    global_step = checkpoint[
        "global_step"
    ]

    best_val_loss = checkpoint[
        "best_val_loss"
    ]

    previous_training_time = checkpoint.get(
        "total_train_time",
        0.0
    )

    if checkpoint.get(
        "mid_epoch",
        True
    ):

        start_epoch = checkpoint[
            "epoch"
        ]

        start_part = checkpoint[
            "part_index"
        ]

        resume_running_loss = checkpoint.get(
            "running_loss",
            0.0
        )

        resume_running_count = checkpoint.get(
            "running_count",
            0
        )

        print()
        print(
            f"Resuming Epoch {start_epoch}"
        )

        print(
            f"Starting from Parquet part "
            f"{start_part:,}"
        )

    else:

        start_epoch = (
            checkpoint["epoch"] + 1
        )

        start_part = 0

        print()
        print(
            f"Resuming from Epoch "
            f"{start_epoch}"
        )

    print(
        f"Global step: "
        f"{global_step:,}"
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.5f}"
    )

else:

    print()
    print(
        "Starting new full training run."
    )

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

full_start = time.time()

for epoch in range(
    start_epoch,
    MAX_EPOCHS + 1
):

    print()
    print("=" * 60)
    print(
        f"EPOCH {epoch}/{MAX_EPOCHS}"
    )
    print("=" * 60)

    # --------------------------------------------------------
    # DATASET FOR CURRENT EPOCH
    # --------------------------------------------------------

    train_dataset = KTSequenceDataset(
        PART_FILES,
        train=True,
        seq_len=SEQ_LEN,
        stride=STRIDE,
        max_parts=None,
        start_part=start_part
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_dataset = KTSequenceDataset(
        PART_FILES,
        train=False,
        seq_len=SEQ_LEN,
        stride=STRIDE,
        max_parts=None,
        start_part=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()

    epoch_start = time.time()

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    (
        train_loss,
        global_step,
        _,
        _
    ) = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        global_step=global_step,
        checkpoint_interval_minutes=SAVE_EVERY_MINUTES,
        best_val_loss=best_val_loss,
        total_train_time=previous_training_time,
        running_loss_start=resume_running_loss,
        running_count_start=resume_running_count
    )

    # --------------------------------------------------------
    # RESET RESUME POSITION
    # --------------------------------------------------------

    start_part = 0
    resume_running_loss = 0.0
    resume_running_count = 0

    print()
    print(
        "Running validation..."
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_metrics = evaluate(
        model,
        val_loader
    )

    epoch_time = (
        time.time()
        - epoch_start
    )

    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    scheduler.step()

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    is_best = (
        val_metrics["loss"]
        < best_val_loss
    )

    if is_best:
        best_val_loss = (
            val_metrics["loss"]
        )

    # --------------------------------------------------------
    # TOTAL TIME
    # --------------------------------------------------------

    total_training_time = (
        previous_training_time
        + time.time()
        - full_start
    )

    previous_training_time = (
        total_training_time
    )

    # --------------------------------------------------------
    # EPOCH CHECKPOINT
    # --------------------------------------------------------

    checkpoint = save_training_checkpoint(
        epoch=epoch,
        global_step=global_step,
        part_index=0,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        best_val_loss=best_val_loss,
        total_train_time=total_training_time,
        mid_epoch=False,
        running_loss=0.0,
        running_count=0
    )

    epoch_checkpoint = (
        FULL_CHECKPOINT_DIR
        / f"epoch_{epoch:02d}.pt"
    )

    torch.save(
        checkpoint,
        epoch_checkpoint
    )

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    if is_best:

        torch.save(
            checkpoint,
            BEST_FULL_CHECKPOINT
        )

        best_status = " <-- BEST MODEL"

    else:

        best_status = ""

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print(
        f"EPOCH {epoch} RESULTS"
    )
    print("=" * 60)

    print(
        f"Training loss       : "
        f"{train_loss:.5f}"
    )

    print(
        f"Validation loss     : "
        f"{val_metrics['loss']:.5f}"
    )

    print(
        f"Validation accuracy : "
        f"{val_metrics['accuracy']:.5f}"
    )

    print(
        f"Validation samples  : "
        f"{val_metrics['count']:,}"
    )

    print(
        f"Epoch time          : "
        f"{epoch_time / 3600:.2f} hours"
    )

    print(
        f"Learning rate       : "
        f"{optimizer.param_groups[0]['lr']:.8f}"
    )

    print(
        f"Checkpoint          : "
        f"{epoch_checkpoint}"
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.5f}"
        f"{best_status}"
    )

    print_gpu_stats()

    # --------------------------------------------------------
    # PAUSE BETWEEN EPOCHS
    # --------------------------------------------------------

    if epoch < MAX_EPOCHS:

        print()
        print("=" * 60)
        print("EPOCH COMPLETE - TRAINING PAUSED")
        print("=" * 60)

        print(
            f"Epoch {epoch} is completely finished."
        )

        print(
            "The checkpoint has been saved."
        )

        print(
            f"Run this cell again when you are "
            f"ready for Epoch {epoch + 1}."
        )

        break

else:

    # --------------------------------------------------------
    # COMPLETE
    # --------------------------------------------------------

    full_time = (
        previous_training_time
    )

    print()
    print("=" * 60)
    print("LEARNINGDNA FULL TRAINING COMPLETE")
    print("=" * 60)

    print(
        f"Total training time : "
        f"{full_time / 3600:.2f} hours"
    )

    print(
        f"Final global step   : "
        f"{global_step:,}"
    )

    print(
        f"Best validation loss: "
        f"{best_val_loss:.5f}"
    )

    print(
        f"Best checkpoint     : "
        f"{BEST_FULL_CHECKPOINT}"
    )

    print_gpu_stats()

LEARNINGDNA FULL TRAINING
Total dataset parts : 15,687
Training epochs     : 3
Batch size          : 64
Sequence length     : 128
Stride              : 64
Model dimension     : 192
Transformer layers  : 3
Attention heads     : 6
Workers             : 0
AMP enabled         : True
Checkpoint interval : 30 minutes

Model created.
Trainable parameters: 1,769,353

FULL TRAINING CHECKPOINT FOUND


D:\Lokesh files\Projects\LearningDNA\venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Saved epoch       : 3
Saved global step : 58,148
Saved part        : 657
Mid-epoch         : True

Resuming Epoch 3
Starting from Parquet part 657
Global step: 58,148
Best validation loss: 0.53551

EPOCH 3/3
Epoch 3 | Batch 5,400 | Global step 63,549 | Loss 0.5237 | Time 0.49 h

MID-EPOCH CHECKPOINT SAVED
Epoch       : 3
Global step : 63,630
Resume part : 3,094
Checkpoint  : D:\Lokesh files\Projects\LearningDNA\models\full_training_checkpoints\latest_full_training.pt
You may safely interrupt the notebook now.

Epoch 3 | Batch 10,700 | Global step 68,849 | Loss 0.5241 | Time 0.99 h

MID-EPOCH CHECKPOINT SAVED
Epoch       : 3
Global step : 68,919
Resume part : 6,327
Checkpoint  : D:\Lokesh files\Projects\LearningDNA\models\full_training_checkpoints\latest_full_training.pt
You may safely interrupt the notebook now.

Epoch 3 | Batch 16,100 | Global step 74,249 | Loss 0.5277 | Time 1.50 h

MID-EPOCH CHECKPOINT SAVED
Epoch       : 3
Global step : 74,279
Resume part : 10,926
Checkpoint  : D:\

In [27]:
# ============================================================
# FINAL MODEL EVALUATION
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    mean_squared_error,
    log_loss,
    confusion_matrix
)

print("=" * 60)
print("LEARNINGDNA FINAL MODEL EVALUATION")
print("=" * 60)

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

BEST_MODEL_PATH = (
    MODEL_DIR
    / "full_training_checkpoints"
    / "best_model.pt"
)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best model not found:\n{BEST_MODEL_PATH}"
    )

print(f"Best model: {BEST_MODEL_PATH}")
print()

# ------------------------------------------------------------
# LOAD BEST CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

print("Checkpoint information:")
print(f"Epoch       : {checkpoint['epoch']}")
print(f"Global step : {checkpoint['global_step']:,}")

if "val_loss" in checkpoint:
    print(
        f"Checkpoint validation loss: "
        f"{checkpoint['val_loss']:.5f}"
    )

if "val_accuracy" in checkpoint:
    print(
        f"Checkpoint validation accuracy: "
        f"{checkpoint['val_accuracy']:.5f}"
    )

print()

# ------------------------------------------------------------
# RECREATE MODEL
# ------------------------------------------------------------

model = LearningDNA(
    num_questions=len(question_to_idx),
    num_skills=len(skill_to_idx),
    question_dim=QUESTION_DIM,
    skill_dim=SKILL_DIM,
    correctness_dim=CORRECTNESS_DIM,
    elapsed_dim=ELAPSED_DIM,
    attempt_dim=ATTEMPT_DIM,
    time_delta_dim=TIME_DELTA_DIM,
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
    max_seq_len=SEQ_LEN
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state"]
)

model.eval()

print("Best model loaded successfully.")
print()

# ------------------------------------------------------------
# FULL VALIDATION DATASET
# ------------------------------------------------------------

val_dataset = KTSequenceDataset(
    PART_FILES,
    train=False,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    max_parts=None,
    start_part=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Full validation loader ready.")
print()

# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

all_probabilities = []
all_targets = []

total_loss = 0.0
total_correct = 0
total_count = 0

eval_start = time.time()

with torch.inference_mode():

    for batch_idx, batch in enumerate(
        val_loader
    ):

        batch = move_batch_to_device(
            batch
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            logits, hidden = model(
                batch["question_idx"],
                batch["skill_idx"],
                batch["correctness_idx"],
                batch["elapsed"],
                batch["attempt_idx"],
                batch["time_delta"],
                batch["target_question_idx"]
            )

        mask = batch["target_mask"]

        if mask.sum() == 0:
            continue

        valid_logits = logits[mask].float()
        valid_targets = batch["target"][mask].float()
        
        # ----------------------------------------------------
        # PROBABILITIES
        # ----------------------------------------------------
        
        probabilities = torch.sigmoid(
            valid_logits
        )
        
        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------
        
        batch_loss = F.binary_cross_entropy_with_logits(
            valid_logits,
            valid_targets,
            reduction="sum"
        )

        total_loss += (
            batch_loss.item()
        )

        # ----------------------------------------------------
        # ACCURACY
        # ----------------------------------------------------

        predictions = (
            probabilities >= 0.5
        ).float()

        total_correct += (
            (predictions == valid_targets)
            .sum()
            .item()
        )

        total_count += (
            valid_targets.numel()
        )

        # ----------------------------------------------------
        # STORE FOR AUC / PR-AUC / RMSE
        # ----------------------------------------------------

        all_probabilities.append(
            probabilities
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        all_targets.append(
            valid_targets
            .detach()
            .float()
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if batch_idx % 100 == 0:

            elapsed = (
                time.time()
                - eval_start
            )

            print(
                f"\r"
                f"Validation batch "
                f"{batch_idx:,} | "
                f"Samples "
                f"{total_count:,} | "
                f"Time "
                f"{elapsed / 60:.1f} min",
                end=""
            )

print()
print()

# ------------------------------------------------------------
# COMBINE RESULTS
# ------------------------------------------------------------

probabilities = np.concatenate(
    all_probabilities
)

targets = np.concatenate(
    all_targets
)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

accuracy = (
    total_correct
    / total_count
)

average_loss = (
    total_loss
    / total_count
)

roc_auc = roc_auc_score(
    targets,
    probabilities
)

pr_auc = average_precision_score(
    targets,
    probabilities
)

rmse = np.sqrt(
    mean_squared_error(
        targets,
        probabilities
    )
)

logloss = log_loss(
    targets,
    probabilities,
    labels=[0, 1]
)

# ------------------------------------------------------------
# CONFUSION MATRIX
# ------------------------------------------------------------

predictions = (
    probabilities >= 0.5
).astype(np.int64)

tn, fp, fn, tp = confusion_matrix(
    targets.astype(np.int64),
    predictions,
    labels=[0, 1]
).ravel()

# ------------------------------------------------------------
# BASELINE
# ------------------------------------------------------------

positive_rate = (
    targets.mean()
)

always_correct_accuracy = (
    positive_rate
)

always_correct_rmse = np.sqrt(
    np.mean(
        (targets - 1.0) ** 2
    )
)

# ------------------------------------------------------------
# FINAL RESULTS
# ------------------------------------------------------------

total_time = (
    time.time()
    - eval_start
)

print("=" * 60)
print("FINAL LEARNINGDNA RESULTS")
print("=" * 60)

print(
    f"Validation samples : "
    f"{total_count:,}"
)

print(
    f"Accuracy           : "
    f"{accuracy:.6f} "
    f"({accuracy * 100:.4f}%)"
)

print(
    f"ROC-AUC            : "
    f"{roc_auc:.6f}"
)

print(
    f"PR-AUC             : "
    f"{pr_auc:.6f}"
)

print(
    f"RMSE               : "
    f"{rmse:.6f}"
)

print(
    f"Log loss           : "
    f"{logloss:.6f}"
)

print(
    f"Binary loss        : "
    f"{average_loss:.6f}"
)

print()
print("Confusion Matrix")
print("----------------")
print(f"True negatives     : {tn:,}")
print(f"False positives    : {fp:,}")
print(f"False negatives    : {fn:,}")
print(f"True positives     : {tp:,}")

print()
print("Baseline")
print("--------")
print(
    f"Always-correct accuracy : "
    f"{always_correct_accuracy:.6f} "
    f"({always_correct_accuracy * 100:.4f}%)"
)

print(
    f"Always-correct RMSE     : "
    f"{always_correct_rmse:.6f}"
)

print()
print(
    f"Evaluation time   : "
    f"{total_time / 60:.2f} minutes"
)

print()
print("=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)

LEARNINGDNA FINAL MODEL EVALUATION
Best model: D:\Lokesh files\Projects\LearningDNA\models\full_training_checkpoints\best_model.pt

Checkpoint information:
Epoch       : 3
Global step : 79,498

Best model loaded successfully.

Full validation loader ready.



D:\Lokesh files\Projects\LearningDNA\venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Validation batch 2,900 | Samples 16,555,645 | Time 6.3 min

FINAL LEARNINGDNA RESULTS
Validation samples : 16,685,316
Accuracy           : 0.732060 (73.2060%)
ROC-AUC            : 0.765653
PR-AUC             : 0.862196
RMSE               : 0.422295
Log loss           : 0.531067
Binary loss        : 0.531067

Confusion Matrix
----------------
True negatives     : 2,305,503
False positives    : 3,242,932
False negatives    : 1,227,734
True positives     : 9,909,147

Baseline
--------
Always-correct accuracy : 0.667466 (66.7466%)
Always-correct RMSE     : 0.576658

Evaluation time   : 6.63 minutes

EVALUATION COMPLETE
